<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [2]:
!pip install folium
!pip install wget
!pip install pandas

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=4db6a6b2404bca62bd60d8e22a752c901927a1061ebf1204deff7de30e2c2a56
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [3]:
import folium
import pandas as pd

In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [16]:
## Task 1: Mark all launch sites on a map

import folium
import pandas as pd
from folium.features import DivIcon
import requests # Import requests for fetching data
import io

# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]

# Download and read the `spacex_launch_geo.csv`
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'

# Use requests to get the content and then io.BytesIO for pandas
resp = requests.get(URL)
spacex_csv_file = io.BytesIO(resp.content)
spacex_df = pd.read_csv(spacex_csv_file)
print(spacex_df.head(5))

# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label

for index, row in launch_sites_df.iterrows():
    # Get coordinates and launch site name
    coordinate = [row['Lat'], row['Long']]
    launch_site_name = row['Launch Site']

    # Create a Circle object
    circle = folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(launch_site_name))

    # Create a Marker object
    marker = folium.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % launch_site_name,
        )
    )

    # Add the circle and marker to the map
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

   Flight Number        Date Time (UTC) Booster Version  Launch Site  \
0              1  2010-06-04   18:45:00  F9 v1.0  B0003  CCAFS LC-40   
1              2  2010-12-08   15:43:00  F9 v1.0  B0004  CCAFS LC-40   
2              3  2012-05-22    7:44:00  F9 v1.0  B0005  CCAFS LC-40   
3              4  2012-10-08    0:35:00  F9 v1.0  B0006  CCAFS LC-40   
4              5  2013-03-01   15:10:00  F9 v1.0  B0007  CCAFS LC-40   

                                             Payload  Payload Mass (kg)  \
0               Dragon Spacecraft Qualification Unit                0.0   
1  Dragon demo flight C1, two CubeSats,  barrel o...                0.0   
2                             Dragon demo flight C2+              525.0   
3                                       SpaceX CRS-1              500.0   
4                                       SpaceX CRS-2              677.0   

       Orbit         Customer        Landing Outcome  class        Lat  \
0        LEO           SpaceX  Failure   (

A continuación, te explico las tres líneas de código que procesan el DataFrame `spacex_df` para crear `launch_sites_df`:

1.  **`spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]`**
    *   **Propósito:** Esta línea filtra el DataFrame original `spacex_df` y selecciona solo las columnas que son relevantes para el análisis: `'Launch Site'` (Sitio de Lanzamiento), `'Lat'` (Latitud), `'Long'` (Longitud) y `'class'` (Clase o resultado del lanzamiento, donde 1 es éxito y 0 es fallo). Las columnas seleccionadas se usan para actualizar el mismo DataFrame `spacex_df`, lo que significa que el DataFrame `spacex_df` ahora solo contiene estas cuatro columnas.

2.  **`launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()`**
    *   **Propósito:** Esta línea agrupa el DataFrame `spacex_df` por la columna `'Launch Site'`. El método `.first()` selecciona la primera fila de cada grupo (es decir, de cada sitio de lanzamiento único). `as_index=False` asegura que `'Launch Site'` se mantenga como una columna normal en lugar de convertirse en el índice del nuevo DataFrame. El resultado es un nuevo DataFrame llamado `launch_sites_df` que contiene una entrada única para cada sitio de lanzamiento, con sus coordenadas y la clase del primer lanzamiento registrado en ese sitio.

3.  **`launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]`**
    *   **Propósito:** Similar a la primera línea, esta línea filtra el `launch_sites_df` recién creado. En este caso, se seleccionan solo las columnas `'Launch Site'`, `'Lat'` y `'Long'`. Esto se hace para asegurar que el `launch_sites_df` final contenga únicamente la información de identificación y ubicación de cada sitio de lanzamiento, eliminando la columna `'class'` que ya no es necesaria para la representación inicial de los sitios en el mapa.

En la línea `for index, row in launch_sites_df.iterrows():`:

*   El método `iterrows()` de un DataFrame de pandas está diseñado para iterar sobre las filas del DataFrame, devolviendo en cada iteración **tanto el índice de la fila como la fila completa** (como un objeto `Series`).
*   Aunque en este bucle específico no se utiliza la variable `index` directamente dentro del cuerpo del bucle, la variable `row` es fundamental. Es `row` la que nos permite acceder a los valores de cada columna para la fila actual, como `row['Lat']` y `row['Long']`, que son las coordenadas que necesitamos para crear los círculos y marcadores en el mapa.
*   Podríamos haber usado `_` en lugar de `index` si no fuéramos a usar el índice (`for _, row in launch_sites_df.iterrows():`), pero usar `index` explícitamente es una práctica común para recordar que `iterrows` proporciona ambos valores.

First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


Now, you can take a look at what are the coordinates for each site.


In [14]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [17]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [18]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [ ]:
# Task 2: Mark the success/failed launches for each site on the map


In [20]:
# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

def get_marker_color(launch_class):
    if launch_class == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(get_marker_color)

In [22]:
marker_cluster = MarkerCluster()
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed,
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # Create and add a Marker cluster to the site map
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color'],
        icon_size=(20, 20),
        html=f'<div style="font-size: 12; color:{record["marker_color"]};"><b>{record["Launch Site"]}</b></div>')
    )
    marker_cluster.add_child(marker)

site_map

El valor `launch_class` dentro de la función `get_marker_color()` proviene de la columna `'class'` del DataFrame `spacex_df`.

Cuando escribes la línea:

```python
spacex_df['marker_color'] = spacex_df['class'].apply(get_marker_color)
```

Está ocurriendo lo siguiente:

1.  **`spacex_df['class']`**: Esto selecciona la columna completa `'class'` del DataFrame `spacex_df`.
2.  **`.apply(get_marker_color)`**: El método `.apply()` es una forma muy útil de aplicar una función (en este caso, `get_marker_color`) a cada elemento de una Serie de pandas (que es lo que es `spacex_df['class']`).
    *   `apply()` itera automáticamente a través de cada valor en la columna `'class'`. Por ejemplo, si el primer valor en `'class'` es `0`, `apply()` llamará a `get_marker_color(0)`. Si el siguiente valor es `1`, llamará a `get_marker_color(1)`, y así sucesivamente.
    *   Cada valor individual de la columna `'class'` se pasa como el argumento `launch_class` a tu función.
3.  **`spacex_df['marker_color'] = ...`**: Finalmente, el resultado que devuelve `get_marker_color` (ya sea `'green'` o `'red'`) para cada fila se guarda en una nueva columna llamada `'marker_color'` en el DataFrame `spacex_df`, en la fila correspondiente.

Entonces, no pasas la columna completa directamente a la función como un solo valor, sino que `apply()` se encarga de pasar cada elemento de la columna uno por uno a la función.

Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [19]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [23]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [ ]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


In [30]:
# Ejemplo: Coordenadas del sitio de lanzamiento 'CCAFS LC-40'
launch_site_lat = 28.562302
launch_site_lon = -80.577356

# Ejemplo: Coordenadas de un punto cercano en la costa (puedes ajustarlas con MousePosition)
coastline_lat = 28.56367
coastline_lon = -80.56794

# Calcular la distancia
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distancia a la costa: {distance_coastline:.2f} KM")

Distancia a la costa: 0.93 KM


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [31]:
# Crea un marcador en el punto de la costa
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html=f'<div style="font-size: 12; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>',
        )
)
site_map.add_child(distance_marker)

# Dibuja una línea entre el sitio de lanzamiento y el punto de la costa
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines = folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

# Muestra el mapa actualizado
site_map

In [24]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [25]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [33]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

In [27]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


Ahora, el mapa mostrará la distancia entre 'CCAFS LC-40' y el punto que usamos para Titusville. Puedes usar la herramienta `MousePosition` para encontrar coordenadas más precisas para cualquier ciudad o punto de interés y luego modificar las variables `city_lat` y `city_lon` en el código anterior para recalcular la distancia y actualizar el mapa.

In [32]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site


### Ejemplo: Distancia a una ciudad (Titusville)

Ahora, sigamos un ejemplo para calcular la distancia entre un sitio de lanzamiento y una ciudad cercana. Usaremos el sitio de lanzamiento 'CCAFS LC-40' y la ciudad de Titusville como ejemplo.

In [34]:
# Coordenadas del sitio de lanzamiento 'CCAFS LC-40'
launch_site_lat_city = 28.562302
launch_site_lon_city = -80.577356

# Coordenadas de ejemplo para la ciudad de Titusville (puedes ajustarlas con MousePosition)
city_lat = 28.5994
city_lon = -80.8078

# Calcular la distancia a la ciudad
distance_city = calculate_distance(launch_site_lat_city, launch_site_lon_city, city_lat, city_lon)
print(f"Distancia a la ciudad (Titusville): {distance_city:.2f} KM")

Distancia a la ciudad (Titusville): 22.88 KM


In [35]:
# Crea un marcador en el punto de la ciudad
distance_marker_city = folium.Marker(
    [city_lat, city_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html=f'<div style="font-size: 12; color:#d35400;"><b>{distance_city:.2f} KM (Titusville)</b></div>',
        )
)
site_map.add_child(distance_marker_city)

# Dibuja una línea entre el sitio de lanzamiento y el punto de la ciudad
city_coordinates = [[launch_site_lat_city, launch_site_lon_city], [city_lat, city_lon]]
lines_city = folium.PolyLine(locations=city_coordinates, weight=1, color='blue')
site_map.add_child(lines_city)

# Muestra el mapa actualizado
site_map

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
